# Scamalyzer Multilingual Training (Colab)

This notebook trains:
1. **DistilBERT** as a separate fine-tuned model for each language (EN, ES, DE)
2. **BiLSTM** per language (EN, ES, DE)
3. **XGBoost + TF-IDF** per language (EN, ES, DE)

It saves artifacts under `/content/output`.

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy tensorflow xgboost joblib

## Configuration

Update the dataset paths if needed. In Colab, upload files to `/content` or mount Drive.

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import joblib
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import gc

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

from keras.models import Sequential
from keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME_BY_LANG = {
    "en": "distilbert-base-uncased",
    "es": "dccuchile/distilbert-base-spanish-uncased",
    "de": "distilbert-base-german-cased",
}
MAX_LEN_BERT = 256
BERT_EPOCHS = 2
BERT_BATCH_SIZE = 16

MAX_LEN_BILSTM = 200
MAX_WORDS_BILSTM = 50000
EMBED_DIM_BILSTM = 100
BILSTM_EPOCHS = 2
BILSTM_BATCH_SIZE = 64

XGB_ESTIMATORS = 50

DATASET_PATHS = {
    "en": "/content/anonymized_dataset.csv",
    "es": "/content/anonymized_dataset_es.csv",
    "de": "/content/anonymized_dataset_de.csv",
}

OUTPUT_DIR = "/content/output"
BERT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "distilbert_finetuned_language_specific")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BERT_OUTPUT_DIR, exist_ok=True)

def plot_cm(cm, title):
    plt.figure(figsize=(4, 3))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Legit (0)", "Fraud (1)"],
        yticklabels=["Legit (0)", "Fraud (1)"],
    )
    plt.title(title)
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

In [ ]:
def load_language_dataset(path, language):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing dataset for {language}: {path}")
    df = pd.read_csv(path)
    if "message" not in df.columns or "label" not in df.columns:
        raise ValueError(f"Dataset {path} must contain message and label columns")
    df = df.dropna(subset=["message", "label"]).copy()
    df["label"] = df["label"].astype(int)
    df["language"] = language
    return df

frames = [load_language_dataset(path, lang) for lang, path in DATASET_PATHS.items()]
df_all = pd.concat(frames, ignore_index=True)
df_all["stratify_key"] = df_all["label"].astype(str) + "_" + df_all["language"]

print("Combined samples:", len(df_all))
print(df_all.groupby(["language", "label"]).size())

## Train DistilBERT Per Language

In [ ]:
def train_bert_for_language(df_lang, lang):
    train_df, test_df = train_test_split(
        df_lang,
        test_size=0.2,
        stratify=df_lang["label"],
        random_state=SEED,
    )
    train_df, val_df = train_test_split(
        train_df,
        test_size=0.125,
        stratify=train_df["label"],
        random_state=SEED,
    )

    model_name = MODEL_NAME_BY_LANG[lang]
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(batch):
        return tokenizer(
            batch["message"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN_BERT,
        )

    train_ds = Dataset.from_pandas(train_df[["message", "label"]]).map(preprocess, batched=True)
    val_ds = Dataset.from_pandas(val_df[["message", "label"]]).map(preprocess, batched=True)
    test_ds = Dataset.from_pandas(test_df[["message", "label"]]).map(preprocess, batched=True)

    for ds in (train_ds, val_ds, test_ds):
        ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds, zero_division=0),
            "f1": f1_score(labels, preds, zero_division=0),
        }

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    # transformers compatibility: some versions use `evaluation_strategy`, others `eval_strategy`
    training_args_kwargs = {
        "output_dir": os.path.join(BERT_OUTPUT_DIR, lang),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": max(4, BERT_BATCH_SIZE // 2),
        "per_device_eval_batch_size": max(4, BERT_BATCH_SIZE // 2),
        "num_train_epochs": max(1, BERT_EPOCHS - 1),
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "save_total_limit": 2,
        "seed": SEED,
        "fp16": torch.cuda.is_available(),
    }

    if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_args_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_args_kwargs["eval_strategy"] = "epoch"
    else:
        print("Warning: no evaluation strategy argument found; evaluation scheduling may default.")

    args = TrainingArguments(**training_args_kwargs)

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_ds,
        "eval_dataset": val_ds,
        "compute_metrics": compute_metrics,
        "callbacks": [EarlyStoppingCallback(early_stopping_patience=2)],
    }

    trainer_init_params = Trainer.__init__.__code__.co_varnames
    if "tokenizer" in trainer_init_params:
        trainer_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in trainer_init_params:
        trainer_kwargs["processing_class"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainer.train()
    eval_metrics = trainer.evaluate(test_ds)

    pred_output = trainer.predict(test_ds)
    logits = pred_output.predictions
    preds = np.argmax(logits, axis=-1)
    y_true = test_df["label"].to_numpy()
    cm = confusion_matrix(y_true, preds)

    model_dir = os.path.join(BERT_OUTPUT_DIR, lang)
    os.makedirs(model_dir, exist_ok=True)
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    del train_ds, val_ds, test_ds, model, trainer, pred_output, logits
    gc.collect()

    return {
        "metrics": eval_metrics,
        "confusion_matrix": cm.tolist(),
        "model_dir": model_dir,
        "base_model": model_name,
    }

bert_results = {}
for lang in ["en", "es", "de"]:
    print(f"Training BERT for {lang} using {MODEL_NAME_BY_LANG[lang]}...")
    df_lang = df_all[df_all["language"] == lang].copy()
    bert_results[lang] = train_bert_for_language(df_lang, lang)
    print(f"  Done. Metrics: {bert_results[lang]['metrics']}")
    gc.collect()

bert_metrics = {lang: payload["metrics"] for lang, payload in bert_results.items()}
bert_conf_matrices = {lang: payload["confusion_matrix"] for lang, payload in bert_results.items()}
print(json.dumps(bert_metrics, indent=2))

## Train BiLSTM Per Language

In [ ]:
def train_bilstm_for_language(df_lang, lang):
    train_df, test_df = train_test_split(
        df_lang, test_size=0.2, stratify=df_lang["label"], random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_df, test_size=0.125, stratify=train_df["label"], random_state=SEED
    )

    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=MAX_WORDS_BILSTM, oov_token="<UNK>")
    tokenizer.fit_on_texts(train_df["message"])

    def prep(texts):
        seqs = tokenizer.texts_to_sequences(texts)
        return tf.keras.preprocessing.sequence.pad_sequences(
            seqs, maxlen=MAX_LEN_BILSTM, padding="post", truncating="post"
        )

    X_train = prep(train_df["message"])
    X_val = prep(val_df["message"])
    X_test = prep(test_df["message"])

    y_train = train_df["label"].values
    y_val = val_df["label"].values
    y_test = test_df["label"].values

    model = Sequential([
        Embedding(input_dim=MAX_WORDS_BILSTM, output_dim=EMBED_DIM_BILSTM, input_length=MAX_LEN_BILSTM),
        Bidirectional(LSTM(128, return_sequences=False)),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")
    ])

    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    es = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=BILSTM_EPOCHS,
        batch_size=BILSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=1
    )

    probs = model.predict(X_test).reshape(-1)
    preds = (probs > 0.5).astype(int)
    cm = confusion_matrix(y_test, preds)

    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
    }

    model_path = os.path.join(OUTPUT_DIR, f"bilstm_model_{lang}.h5")
    tokenizer_path = os.path.join(OUTPUT_DIR, f"bilstm_tokenizer_{lang}.json")
    model.save(model_path)
    with open(tokenizer_path, "w", encoding="utf-8") as f:
        f.write(tokenizer.to_json())

    return {
        "metrics": metrics,
        "confusion_matrix": cm.tolist(),
    }

bilstm_results = {}
for lang in ["en", "es", "de"]:
    df_lang = df_all[df_all["language"] == lang].copy()
    bilstm_results[lang] = train_bilstm_for_language(df_lang, lang)

bilstm_metrics = {lang: payload["metrics"] for lang, payload in bilstm_results.items()}
bilstm_conf_matrices = {lang: payload["confusion_matrix"] for lang, payload in bilstm_results.items()}
print(json.dumps(bilstm_metrics, indent=2))

## Train XGBoost Per Language

In [ ]:
def train_xgb_for_language(df_lang, lang):
    train_df, test_df = train_test_split(
        df_lang, test_size=0.2, stratify=df_lang["label"], random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_df, test_size=0.125, stratify=train_df["label"], random_state=SEED
    )

    # Memory-efficient TF-IDF: reduced features to avoid OOM on 12GB Colab
    # Reduced from max_features=50000 to 10000, and ngram_range=(3,5) to (3,4)
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=10000)
    X_train = tfidf.fit_transform(train_df["message"])
    X_val = tfidf.transform(val_df["message"])
    X_test = tfidf.transform(test_df["message"])

    y_train = train_df["label"].values
    y_val = val_df["label"].values
    y_test = test_df["label"].values

    # Single-threaded to reduce memory overhead, disable early stopping eval_set for memory
    clf = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=XGB_ESTIMATORS,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=1,  # Changed from -1: single-threaded uses less memory
        tree_method="hist"  # Faster, more memory-efficient tree building
    )

    # Train without eval_set to save memory
    clf.fit(X_train, y_train, verbose=False)
    
    # Make predictions and clean up
    probs = clf.predict_proba(X_test)[:, 1]
    preds = (probs > 0.5).astype(int)
    cm = confusion_matrix(y_test, preds)

    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
    }

    model_path = os.path.join(OUTPUT_DIR, f"xgb_model_{lang}.joblib")
    tfidf_path = os.path.join(OUTPUT_DIR, f"tfidf_{lang}.joblib")
    joblib.dump(clf, model_path)
    joblib.dump(tfidf, tfidf_path)
    
    # Explicit cleanup
    del X_train, X_val, X_test, clf, tfidf
    gc.collect()

    return {
        "metrics": metrics,
        "confusion_matrix": cm.tolist(),
    }

xgb_results = {}
for lang in ["en", "es", "de"]:
    print(f"Training XGBoost for {lang}...")
    df_lang = df_all[df_all["language"] == lang].copy()
    xgb_results[lang] = train_xgb_for_language(df_lang, lang)
    print(f"  Done. Metrics: {xgb_results[lang]['metrics']}")
    gc.collect()  # Free memory between language iterations

xgb_metrics = {lang: payload["metrics"] for lang, payload in xgb_results.items()}
xgb_conf_matrices = {lang: payload["confusion_matrix"] for lang, payload in xgb_results.items()}
print(json.dumps(xgb_metrics, indent=2))

## Summary and Artifact List

In [ ]:
print("BERT models:")
for lang in ["en", "es", "de"]:
    if lang in bert_results:
        print(f"- {lang}: {bert_results[lang]['base_model']} -> {bert_results[lang]['model_dir']}")

print("BERT metrics by language:")
print(json.dumps(bert_metrics, indent=2))
print("BiLSTM metrics by language:")
print(json.dumps(bilstm_metrics, indent=2))
print("XGBoost metrics by language:")
print(json.dumps(xgb_metrics, indent=2))

print("\nSaved files:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print("-", name)

print("\nConfusion Matrices: BERT")
for lang in ["en", "es", "de"]:
    if lang in bert_conf_matrices:
        plot_cm(np.array(bert_conf_matrices[lang]), f"BERT - {lang.upper()}")

print("\nConfusion Matrices: BiLSTM")
for lang in ["en", "es", "de"]:
    if lang in bilstm_conf_matrices:
        plot_cm(np.array(bilstm_conf_matrices[lang]), f"BiLSTM - {lang.upper()}")

print("\nConfusion Matrices: XGBoost")
for lang in ["en", "es", "de"]:
    if lang in xgb_conf_matrices:
        plot_cm(np.array(xgb_conf_matrices[lang]), f"XGBoost - {lang.upper()}")

## Model Evaluation: Confusion Matrices, Inference Times & Stats

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import torch

# Load test data
test_data_by_lang = {}
for lang in ["en", "es", "de"]:
    df = pd.read_csv(DATASET_PATHS[lang])
    df = df.dropna(subset=["message", "label"]).copy()
    df["label"] = df["label"].astype(int)
    df["language"] = lang
    
    # Use up to 2000 samples for comprehensive evaluation
    test_sample = df.sample(n=min(2000, len(df)), random_state=SEED)
    test_data_by_lang[lang] = {
        "texts": test_sample["message"].tolist(),
        "labels": test_sample["label"].values,
    }

# ========== DISTILBERT INFERENCE & STATS ==========
print("=" * 80)
print("DISTILBERT INFERENCE TIMES & CONFUSION MATRICES")
print("=" * 80)

bert_inference_times = {}
for lang in ["en", "es", "de"]:
    model_path = os.path.join(BERT_OUTPUT_DIR, lang)
    if not os.path.exists(model_path):
        print(f"  Skipping {lang}: model not found at {model_path}")
        continue
    
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model.eval()
    
    texts = test_data_by_lang[lang]["texts"]
    true_labels = test_data_by_lang[lang]["labels"]
    
    # Inference timing
    start_time = time.time()
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1).numpy()
    inference_time = time.time() - start_time
    
    bert_inference_times[lang] = inference_time / len(texts)
    
    cm = confusion_matrix(true_labels, preds)
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, zero_division=0)
    rec = recall_score(true_labels, preds, zero_division=0)
    f1 = f1_score(true_labels, preds, zero_division=0)
    
    print(f"\n{lang.upper()}:")
    print(f"  Samples: {len(texts)}")
    print(f"  Avg inference time: {bert_inference_times[lang]:.4f}s per sample")
    print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    plot_cm(cm, f"DistilBERT - {lang.upper()}")
    del model, tokenizer
    gc.collect()

# ========== BILSTM INFERENCE & STATS ==========
print("\n" + "=" * 80)
print("BILSTM INFERENCE TIMES & CONFUSION MATRICES")
print("=" * 80)

bilstm_inference_times = {}
for lang in ["en", "es", "de"]:
    model_path = os.path.join(OUTPUT_DIR, f"bilstm_model_{lang}.h5")
    tokenizer_path = os.path.join(OUTPUT_DIR, f"bilstm_tokenizer_{lang}.json")
    
    if not os.path.exists(model_path) or not os.path.exists(tokenizer_path):
        print(f"  Skipping {lang}: model/tokenizer not found")
        continue
    
    model = tf.keras.models.load_model(model_path)
    with open(tokenizer_path, "r", encoding="utf-8") as f:
        tokenizer_config = json.load(f)
        tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_config)
    
    texts = test_data_by_lang[lang]["texts"]
    true_labels = test_data_by_lang[lang]["labels"]
    
    # Preprocess
    seqs = tokenizer.texts_to_sequences(texts)
    X = tf.keras.preprocessing.sequence.pad_sequences(seqs, maxlen=200, padding="post", truncating="post")
    
    # Inference timing
    start_time = time.time()
    probs = model.predict(X, verbose=0).reshape(-1)
    inference_time = time.time() - start_time
    preds = (probs > 0.5).astype(int)
    
    bilstm_inference_times[lang] = inference_time / len(texts)
    
    cm = confusion_matrix(true_labels, preds)
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, zero_division=0)
    rec = recall_score(true_labels, preds, zero_division=0)
    f1 = f1_score(true_labels, preds, zero_division=0)
    
    print(f"\n{lang.upper()}:")
    print(f"  Samples: {len(texts)}")
    print(f"  Avg inference time: {bilstm_inference_times[lang]:.4f}s per sample")
    print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    plot_cm(cm, f"BiLSTM - {lang.upper()}")
    del model, tokenizer
    gc.collect()

# ========== XGBOOST INFERENCE & STATS ==========
print("\n" + "=" * 80)
print("XGBOOST INFERENCE TIMES & CONFUSION MATRICES")
print("=" * 80)

xgb_inference_times = {}
for lang in ["en", "es", "de"]:
    model_path = os.path.join(OUTPUT_DIR, f"xgb_model_{lang}.joblib")
    tfidf_path = os.path.join(OUTPUT_DIR, f"tfidf_{lang}.joblib")
    
    if not os.path.exists(model_path) or not os.path.exists(tfidf_path):
        print(f"  Skipping {lang}: model/tfidf not found")
        continue
    
    clf = joblib.load(model_path)
    tfidf = joblib.load(tfidf_path)
    
    texts = test_data_by_lang[lang]["texts"]
    true_labels = test_data_by_lang[lang]["labels"]
    
    # Inference timing
    start_time = time.time()
    X = tfidf.transform(texts)
    probs = clf.predict_proba(X)[:, 1]
    inference_time = time.time() - start_time
    preds = (probs > 0.5).astype(int)
    
    xgb_inference_times[lang] = inference_time / len(texts)
    
    cm = confusion_matrix(true_labels, preds)
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, zero_division=0)
    rec = recall_score(true_labels, preds, zero_division=0)
    f1 = f1_score(true_labels, preds, zero_division=0)
    
    print(f"\n{lang.upper()}:")
    print(f"  Samples: {len(texts)}")
    print(f"  Avg inference time: {xgb_inference_times[lang]:.4f}s per sample")
    print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    plot_cm(cm, f"XGBoost - {lang.upper()}")
    del clf, tfidf
    gc.collect()

# ========== SUMMARY TABLE ==========
print("\n" + "=" * 80)
print("INFERENCE TIME SUMMARY (seconds per sample)")
print("=" * 80)
summary_df = pd.DataFrame({
    "DistilBERT": bert_inference_times,
    "BiLSTM": bilstm_inference_times,
    "XGBoost": xgb_inference_times,
})
print(summary_df.to_string())
print("\nFastest model per language:")
for lang in ["en", "es", "de"]:
    if lang in summary_df.index:
        fastest = summary_df.loc[lang].idxmin()
        fastest_time = summary_df.loc[lang].min()
        print(f"  {lang.upper()}: {fastest} ({fastest_time:.4f}s)")